In [3]:
import pandas as pd
import json
import openpyxl
from pathlib import Path

In [4]:
def find_data_dir():
    current_dir = Path.cwd().resolve()
    search_roots = [current_dir, *current_dir.parents]

    for root in search_roots:
        data_dir = root / "data" / "xlsx"
        if (data_dir / "DPP_FIBROTOR_ER15_V2.xlsx").exists():
            return data_dir

    if (current_dir / "DPP_FIBROTOR_ER15_V2.xlsx").exists():
        return current_dir

    raise FileNotFoundError("Could not find DPP_FIBROTOR_ER15_V2.xlsx relative to " + str(current_dir))

DATA_DIR = find_data_dir()
asset_file = DATA_DIR / "DPP_FIBROTOR_ER15_V2.xlsx"

wb = openpyxl.load_workbook(asset_file, data_only=True)
wb_names = wb.sheetnames
print(wb.sheetnames)

['Digital Nameplate', 'Handover Documentation', 'Technical Data', 'Carbon Footprint', 'Maintenance Instructions', 'Sheet1']


In [5]:
with open(DATA_DIR / "IDTA 02006-3-0-1_Template_Digital Nameplate.json") as f:
    digital_nameplate = json.load(f)

with open(DATA_DIR / "IDTA 02004-2-0_Template_HandoverDocumentation.json") as f:
    handover_documentation = json.load(f)

with open(DATA_DIR / "IDTA 02003_Template_TechnicalData.json") as f:
    technical_data = json.load(f)

with open(DATA_DIR / "IDTA 02023 _Template_CarbonFootprint.json") as f:
    carbon_footprint = json.load(f)

with open(DATA_DIR / "IDTA_02018_Template_MaintenanceInstructions.json") as f:
    maintenance_instructions = json.load(f)

In [ ]:
from datetime import date, datetime, timedelta

def coerce_value(value, value_type: str) -> str:
    """Coerce Excel values into AAS-compliant string representations."""
    if value is None or str(value).strip() in ("", "N/A", "__MISSING__"):
        return None

    value_type_lower = value_type.lower()

    if "date" in value_type_lower:
        if isinstance(value, datetime):
            value = value.date()
        if isinstance(value, date):
            date_value = value.strftime("%Y-%m-%d")
            return f"{date_value}T00:00:00" if "datetime" in value_type_lower else date_value

    val_str = str(value).strip()

    if "boolean" in value_type_lower:
        if val_str.lower() in ("true", "1", "yes", "y", "true.0"):
            return "true"
        return "false"

    if any(x in value_type_lower for x in ("integer", "int", "long", "short")):
        try:
            return str(int(float(val_str)))
        except ValueError:
            return val_str

    if "date" in value_type_lower:
        for fmt in ("%Y-%m-%d", "%Y-%m-%d %H:%M:%S", "%d/%m/%Y", "%m/%d/%Y"):
            try:
                parsed = datetime.strptime(val_str, fmt)
                date_value = parsed.strftime("%Y-%m-%d")
                return f"{date_value}T00:00:00" if "datetime" in value_type_lower else date_value
            except ValueError:
                pass
        try:
            serial = float(val_str)
            parsed = datetime(1899, 12, 30) + timedelta(days=serial)
            date_value = parsed.strftime("%Y-%m-%d")
            return f"{date_value}T00:00:00" if "datetime" in value_type_lower else date_value
        except ValueError:
            return val_str

    if any(x in value_type_lower for x in ("double", "float", "decimal")):
        try:
            return str(float(val_str))
        except ValueError:
            return val_str

    return val_str

In [7]:
def extract_table(rows, header_row_idx, key_idx, val_idx, ob_idx, sheet_name, missing_mandatory_log):
    """
    Extracts key-value pairs from a table using pre-located column indexes.
    Returns the extracted data and the index of the row where extraction stopped.
    """
    # print(rows[header_row_idx], key_idx, val_idx, ob_idx, sheet_name)
    data = {}
    row_idx = header_row_idx + 1  # Start reading data from the row after the header
    
    while row_idx < len(rows):
        row = rows[row_idx]
        
        # Stop if the row is empty or doesn't have a key cell
        if not row or len(row) <= key_idx:
            break
            
        key = row[key_idx]
        # print("the key is", key, row)
        if key is None or str(key).strip() == "":
            break  # End of this table
            
        key = str(key).strip()
        
        # Retrieve values using determined indices safely
        value = row[val_idx] if val_idx is not None and val_idx < len(row) else None
        obligation = row[ob_idx] if ob_idx is not None and ob_idx < len(row) else None
        # print("the value is", value,"the index of the value is", val_idx, "the obligation is", obligation)
        if isinstance(value, str):
            value = value.strip()

        is_empty = value is None or str(value).strip() in ("", "N/A")
        
        if obligation == "Mandatory" and is_empty:
            data[key] = "__MISSING__"
            missing_mandatory_log.append(f"{sheet_name} -> {key}")
        else:
            data[key] = None if is_empty else value
            
        row_idx += 1
        
    return data, row_idx

In [8]:
def find_submodels(sheet, missing_mandatory_log):
    """
    Scans the sheet for tables starting with a header row containing 'idShort'.
    Extracts them into a scoped dictionary.
    """
    submodels = {}
    rows = list(sheet.iter_rows(values_only=True))
    
    i = 0
    while i < len(rows):
        row = rows[i]
        if not row:
            i += 1
            continue
        
        # Look for the presence of "idShort" or "Element Name (idShort)" in the row
        key_idx = next((idx for idx, cell in enumerate(row) if cell and "Element Name (idShort)" in str(cell)), None)
        
        if key_idx is not None:
            # Table/Submodel name is typically in the row directly above (i-1)
            name = "Unknown"
            if i > 0:
                prev_row = rows[i-1]
                if prev_row:
                    # Safely grab the first non-empty text in the row above as the table name
                    name_cell = next((cell for cell in prev_row if cell is not None and str(cell).strip() != ""), "Unknown")
                    name = str(name_cell).strip()
            
            # Dynamically locate "Value" and "Obligation" columns in this header row
            val_idx = next((idx for idx, cell in enumerate(row) if cell and str(cell).strip() == "Actual Value"), None)
            ob_idx = next((idx for idx, cell in enumerate(row) if cell and str(cell).strip() == "Obligation"), None)
            
            # Extract this table's data and get the index of where the table ends
            table_data, end_row_idx = extract_table(
                rows=rows,
                header_row_idx=i,
                key_idx=key_idx,
                val_idx=val_idx,
                ob_idx=ob_idx,
                sheet_name=sheet.title,
                missing_mandatory_log=missing_mandatory_log
            )
            
            submodels[name] = table_data
            i = end_row_idx  # Resume scanning the sheet *after* this table
        else:
            i += 1
            
    return submodels

In [ ]:
# AAS has no standard "dummy value" flag. Use Qualifiers to disclose values that
# satisfy a template's required shape but are not authoritative DPP data.
# Keep the IDTA idShort, semanticId, and valueType unchanged for template conformance.
DPP_VALUE_DISCLOSURES = {
    "Carbon Footprint": {
        "ProductCarbonFootprints": {
            "status": "Placeholder",
            "reason": "Product Carbon Footprint is not calculated for this product; values are prototype placeholders and are not authoritative DPP data.",
        },
        "PcfCO2eq": {
            "value": 0.0,
            "status": "Placeholder",
            "reason": "This CO2-equivalent value is a placeholder until a verified PCF calculation is available.",
        },
    },
}


def apply_value_disclosures(elements: list, disclosures: dict) -> None:
    """Attach AAS Qualifiers that make non-authoritative values explicit."""
    for element in elements:
        if not isinstance(element, dict):
            continue

        disclosure = disclosures.get(element.get("idShort"))
        if disclosure:
            if "value" in disclosure and element.get("modelType") == "Property":
                value_type = element.get("valueType", "xs:string")
                element["value"] = coerce_value(disclosure["value"], value_type)

            qualifiers = element.setdefault("qualifiers", [])
            qualifiers[:] = [
                qualifier for qualifier in qualifiers
                if qualifier.get("type") not in {"DppValueStatus", "DppValueStatusNote"}
            ]
            qualifiers.extend([
                {"type": "DppValueStatus", "valueType": "xs:string", "value": disclosure["status"]},
                {"type": "DppValueStatusNote", "valueType": "xs:string", "value": disclosure["reason"]},
            ])

        nested_elements = element.get("value")
        if element.get("modelType") in ("SubmodelElementCollection", "SubmodelElementList") and isinstance(nested_elements, list):
            apply_value_disclosures(nested_elements, disclosures)


def fill_template(elements: list, company_data: dict, path: tuple = ()) -> None:
    """Recursively fill AAS template elements using sheet-scoped data."""
    for element in elements:
        if not isinstance(element, dict):
            continue

        model = element.get("modelType")
        id_short = element.get("idShort")
        if not id_short:
            continue

        current_path = path + (id_short,)
        path_str = ".".join(current_path)
        value = company_data.get(path_str, company_data.get(id_short))

        if model == "Property":
            if value is not None and value != "__MISSING__":
                val_type = element.get("valueType", "xs:string")
                coerced = coerce_value(value, val_type)
                if coerced is not None:
                    element["value"] = coerced
            elif value == "__MISSING__":
                element.pop("value", None)

        elif model == "MultiLanguageProperty":
            if value is not None and value != "__MISSING__":
                if "value" not in element or not isinstance(element["value"], list):
                    element["value"] = []
                if len(element["value"]) == 0:
                    element["value"].append({"language": "en", "text": ""})
                element["value"][0]["text"] = str(value)
            elif value == "__MISSING__":
                element.pop("value", None)

        elif model == "File":
            if value is not None and value != "__MISSING__" and str(value).strip():
                element["value"] = str(value).strip()
            else:
                element.pop("value", None)

        elif model == "Range":
            if isinstance(value, dict):
                element["min"] = str(value.get("min", ""))
                element["max"] = str(value.get("max", ""))
            elif isinstance(value, (list, tuple)) and len(value) == 2:
                element["min"] = str(value[0])
                element["max"] = str(value[1])

        elif model in ("SubmodelElementCollection", "SubmodelElementList"):
            nested_elements = element.get("value", [])
            if isinstance(nested_elements, list):
                fill_template(nested_elements, company_data, current_path)

In [ ]:
import json

template_registry = {
    "Digital Nameplate": "IDTA 02006-3-0-1_Template_Digital Nameplate.json",
    "Handover Documentation": "IDTA 02004-2-0_Template_HandoverDocumentation.json",
    "Technical Data": "IDTA 02003_Template_TechnicalData.json",
    "Carbon Footprint": "IDTA 02023 _Template_CarbonFootprint.json",
    "Maintenance Instructions": "IDTA_02018_Template_MaintenanceInstructions.json"
}

all_missing_mandatory = []

for sheet_name, template_filename in template_registry.items():
    if sheet_name not in wb.sheetnames:
        print(f"Skipping {sheet_name}: Not found in workbook.")
        continue

    sheet = wb[sheet_name]
    submodels = find_submodels(sheet, all_missing_mandatory)

    sheet_data = {}
    for table_data in submodels.values():
        sheet_data.update(table_data)

    template_path = DATA_DIR / template_filename
    try:
        with open(template_path, "r", encoding="utf-8") as f:
            template_json = json.load(f)
    except FileNotFoundError:
        print(f"Error: Template file not found -> {template_path}")
        continue

    if "submodels" in template_json and len(template_json["submodels"]) > 0:
        submodel_elements = template_json["submodels"][0].get("submodelElements", [])
        fill_template(submodel_elements, sheet_data)
        apply_value_disclosures(submodel_elements, DPP_VALUE_DISCLOSURES.get(sheet_name, {}))

    output_filename = f"output_{sheet_name.replace(' ', '_').lower()}.json"
    output_path = DATA_DIR / output_filename
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(template_json, f, indent=4, ensure_ascii=False)
    print(f"Successfully generated: {output_filename}")

if all_missing_mandatory:
    print(f"\nValidation Warning: {len(all_missing_mandatory)} mandatory fields are missing:")
    for item in all_missing_mandatory:
        print(f"  - {item}")
else:
    print("\nAll mandatory fields successfully populated!")

In [ ]:
import copy
import json

generated_submodels = {
    "carbon-footprint": ("output_carbon_footprint.json", "CarbonFootprint"),
    "technical-data": ("output_technical_data.json", "TechnicalData"),
    "digital-nameplate": ("output_digital_nameplate.json", "DigitalNameplate"),
    "handover-documentation": ("output_handover_documentation.json", "HandoverDocumentation"),
    "maintenance-instructions": ("output_maintenance_instructions.json", "MaintenanceInstructions"),
}

def load_environment(path):
    with open(DATA_DIR / path, "r", encoding="utf-8") as f:
        return json.load(f)

def first_submodel(environment, submodel_id, id_short):
    submodel = copy.deepcopy(environment["submodels"][0])
    submodel["id"] = submodel_id
    submodel["idShort"] = id_short
    return submodel

def merge_concept_descriptions(environments):
    merged = {}
    for environment in environments:
        for concept in environment.get("conceptDescriptions", []):
            concept_id = concept.get("id") or concept.get("idShort")
            if concept_id and concept_id not in merged:
                merged[concept_id] = concept
    return list(merged.values())

environments = {slug: load_environment(filename) for slug, (filename, _) in generated_submodels.items()}
nameplate = environments["digital-nameplate"]
uri_of_the_product = nameplate["submodels"][0]["submodelElements"][0]["value"]
print(uri_of_the_product)

submodels = []
submodel_refs = []
for slug, (_, id_short) in generated_submodels.items():
    submodel_id = f"{uri_of_the_product}/submodels/{slug}"
    submodels.append(first_submodel(environments[slug], submodel_id, id_short))
    submodel_refs.append({
        "type": "ModelReference",
        "keys": [{"type": "Submodel", "value": submodel_id}]
    })

aas_environment = {
    "assetAdministrationShells": [{
        "id": f"{uri_of_the_product}/aas",
        "idShort": "AssetAdministrationShell",
        "modelType": "AssetAdministrationShell",
        "assetInformation": {
            "assetKind": "Instance",
            "globalAssetId": uri_of_the_product
        },
        "submodels": submodel_refs
    }],
    "submodels": submodels,
    "conceptDescriptions": merge_concept_descriptions(environments.values())
}

output_filename = "final_basyx_aas_environment.json"
output_path = DATA_DIR / output_filename
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(aas_environment, f, indent=2, ensure_ascii=False)

print(f"Successfully generated {output_filename} linked to {uri_of_the_product}")

In [ ]:
import requests

from basyx.aas import model
from basyx.aas.adapter import json as aas_json
from basyx.aas.adapter.aasx import (
    AASXWriter,
    DictSupplementaryFileContainer,
)

JSON_FILE = DATA_DIR / output_filename
AASX_FILE = DATA_DIR / "finaltemplate.aasx"
IMAGE_PATH = DATA_DIR / "product.png"
UPLOAD_URL = "http://localhost:8081/upload"

with open(JSON_FILE, "r", encoding="utf-8") as f:
    try:
        object_store = aas_json.read_aas_json_file(f)
    except Exception as e:
        print(f"Error reading AAS JSON file: {e}")
        raise

print("Environment loaded.")

file_store = DictSupplementaryFileContainer()

aas_ids = [
    obj.id
    for obj in object_store
    if isinstance(obj, model.AssetAdministrationShell)
]

print("Found AAS IDs:")
for aid in aas_ids:
    print(" -", aid)

with AASXWriter(AASX_FILE) as writer:
    writer.write_aas(
        aas_ids=aas_ids,
        object_store=object_store,
        file_store=file_store,
    )

print(f"AASX created: {AASX_FILE}")

headers = {"Accept": "application/json"}

with open(AASX_FILE, "rb") as f:
    response = requests.post(
        UPLOAD_URL,
        files={"file": (AASX_FILE.name, f, "application/octet-stream")},
        headers=headers,
    )

print("Status:", response.status_code)
print(response.text)

if response.status_code in (200, 201):
    print("Upload successful!")
else:
    print("Upload failed.")